**MAESTRÍA EN INTELIGENCIA ARTIFICIAL APLICADA**

**Curso: MLOPS**

Tecnológico de Monterrey


**Proyecto Entrega 1**


---

# ***PREPROCESAMIENTO Y FEATURE ENGINEERING***

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import PowerTransformer, MinMaxScaler
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor
from scipy.stats import randint, uniform
import pandas as pd
import numpy as np


In [ ]:
#NECESARIO TENER DF_CLEAN
# 1) Extraer fecha y título desde 'url'
df_clean['url_cleaned'] = df_clean['url'].astype(str).str.strip()  # normaliza: aseguro string y quito espacios

date_pattern = r'/(\d{4})/(\d{2})/(\d{2})/'                        # patrón YYYY/MM/DD dentro de la URL
date_match = df_clean['url_cleaned'].str.extract(date_pattern)     # extrae año/mes/día como columnas 0,1,2

df_clean['article_year']  = pd.to_numeric(date_match[0], errors='coerce')  # convierte año a num (NaN si falla)
df_clean['article_month'] = pd.to_numeric(date_match[1], errors='coerce')  # convierte mes a num
df_clean['article_day']   = pd.to_numeric(date_match[2], errors='coerce')  # convierte día a num

df_clean['article_title'] = df_clean['url_cleaned'].str.split('/').str[-2] # toma el penúltimo segmento como “título”
df_clean['article_title'] = df_clean['article_title'].str.replace('-', ' ').str.title()  # limpia guiones y capitaliza

# Imputación simple para fechas faltantes (moda = valor más frecuente)
df_clean['article_year'].fillna(df_clean['article_year'].mode().iloc[0], inplace=True)
df_clean['article_month'].fillna(df_clean['article_month'].mode().iloc[0], inplace=True)
df_clean['article_day'].fillna(df_clean['article_day'].mode().iloc[0], inplace=True)

print("Nuevas columnas 'article_year', 'article_month', 'article_day' y 'article_title' creadas y modificadas.")
display(df_clean[['url', 'article_year', 'article_month', 'article_day', 'article_title']].head())  

Nuevas columnas 'article_year', 'article_month', 'article_day' y 'article_title' creadas y modificadas.


,url,article_year,article_month,article_day,article_title
0,http://mashable.com/2013/01/07/amazon-instant-...,2013.0,1.0,7.0,Amazon Instant Video Browser
1,http://mashable.com/2013/01/07/ap-samsung-spon...,2013.0,1.0,7.0,Ap Samsung Sponsored Tweets
2,http://mashable.com/2013/01/07/apple-40-billio...,2013.0,1.0,7.0,Apple 40 Billion App Downloads
3,http://mashable.com/2013/01/07/astronaut-notre...,2013.0,1.0,7.0,Astronaut Notre Dame Bcs
4,http://mashable.com/2013/01/07/att-u-verse-apps/,2013.0,1.0,7.0,Att U Verse Apps


In [ ]:
# 2) Definición de columnas (booleanas, no predictoras y target)
boolean_cols = [
    'data_channel_is_lifestyle','data_channel_is_entertainment','data_channel_is_bus',
    'data_channel_is_socmed','data_channel_is_tech','data_channel_is_world',
    'weekday_is_monday','weekday_is_tuesday','weekday_is_wednesday',
    'weekday_is_thursday','weekday_is_friday','weekday_is_saturday',
    'weekday_is_sunday','is_weekend'
]
non_predictors = ['url','article_title','url_cleaned','mixed_type_col']  # columnas a excluir de X
target_col = 'shares'                                                    # variable objetivo

# 3) Copia de trabajo (post-limpieza)
df_model = df_clean.copy()  # evita modificar el original

# Asegura que las booleanas sean 0/1 enteros (no floats)
for c in boolean_cols:
    if c in df_model.columns:
        df_model[c] = (df_model[c] > 0).astype(int)



In [ ]:
# 4) Separación Train/Test
X = df_model.drop(columns=[target_col] + [c for c in non_predictors if c in df_model.columns], errors="ignore")  # features
y = df_model[target_col].copy()  # target

X_train, X_test, y_train, y_test = train_test_split(  # split reproducible
    X, y,
    test_size=0.2,            # 80/20
    random_state=42,          # semilla fija
    shuffle=True              # mezcla filas antes de dividir
)

In [ ]:
# 5) ColumnTransformer (numéricas con Yeo-Johnson + MinMax; booleanas passthrough)
bool_cols_present = [c for c in boolean_cols if c in X_train.columns]                    # booleans reales en X
num_all = X_train.select_dtypes(include=[np.number]).columns.tolist()                    # numéricas
num_all = [c for c in num_all if c not in bool_cols_present]                             # quita las booleanas

# Separamos numéricas “transformables” (no constantes o binarias) de las que pasan sin transformar
transformable_num, passthrough_num = [], []
for c in num_all:
    vals = X_train[c].dropna()
    if vals.nunique() > 2 and vals.std() > 0:  # >2 valores y varianza > 0 → apta para Yeo-Johnson
        transformable_num.append(c)
    else:
        passthrough_num.append(c)

In [ ]:
# Pipeline numérico: Yeo–Johnson (reduce sesgo y maneja negativos) + MinMax (escala 0–1)
num_pipe = Pipeline(steps=[
    ("yeojohnson", PowerTransformer(method="yeo-johnson", standardize=False)),
    ("minmax", MinMaxScaler())
])

# Ensamble final de preprocesamiento por tipo de columna
preprocess = ColumnTransformer(
    transformers=[
        ("num",   num_pipe,          transformable_num),  # aplica transformaciones
        ("num_pt","passthrough",     passthrough_num),    # deja igual constantes/binarias numéricas
        ("bool",  "passthrough",     bool_cols_present),  # deja igual las 0/1
    ],
    remainder="drop"  # ignora cualquier columna no listada
)


In [ ]:
# 6) Funciones auxiliares (métricas y creador de pipeline con TTR)
def eval_real(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))  # raíz del MSE
    mae  = mean_absolute_error(y_true, y_pred)          # error absoluto medio
    r2   = r2_score(y_true, y_pred)                     # varianza explicada
    return {"RMSE": rmse, "MAE": mae, "R2": r2}

def make_pipe(estimator):
    return Pipeline(steps=[
        ("prep", preprocess),                              # preprocesa X
        ("model", TransformedTargetRegressor(              # transforma y en log1p durante el fit
            regressor=estimator,
            func=np.log1p, inverse_func=np.expm1          # invierte a escala real en predict
        ))
    ])

# 7) Esquema de validación cruzada (para búsquedas y evaluación estable)
cv = KFold(n_splits=5, shuffle=True, random_state=42)  # 5 folds estratificados por orden aleatorio

# ***ENTRENAMIENTO (TUNING) DE MODELOS***

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
from xgboost import XGBRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

In [ ]:
# Modelo base HGB con Poisson 
hgb_poisson = HistGradientBoostingRegressor(
    loss="poisson",             # para conteos (shares >= 0)
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    tol=1e-4,
    max_bins=255,
    random_state=42
)

# Pipeline: preprocesamiento + modelo Poisson (sin TTR)
pipe_hgb_pois = Pipeline(steps=[
    ("prep", preprocess),      #no se usa TransformedTargetRegressor
    ("model", hgb_poisson)
])

# Espacio de búsqueda (realista y compacto)
param_hgb = {
    "model__learning_rate": loguniform(0.01, 0.12),
    "model__max_iter": randint(200, 600),
    "model__max_depth": randint(3, 10),
    "model__max_leaf_nodes": randint(24, 140),
    "model__min_samples_leaf": randint(15, 120),
    "model__l2_regularization": loguniform(1e-9, 1e-2),
}

search_hgb = RandomizedSearchCV(
    estimator=pipe_hgb_pois,
    param_distributions=param_hgb,
    n_iter=40,                      
    scoring="neg_mean_absolute_error",
    cv=cv,                          #KFold(n_splits=5, shuffle=True, random_state=42)
    n_jobs=-1,
    random_state=42,
    verbose=1
)

# Entrenar búsqueda
search_hgb.fit(X_train, y_train)

print("Best params (HGB-Poisson):", search_hgb.best_params_)
print("Best CV MAE:", -search_hgb.best_score_)

# Evaluación en test (sin transformaciones del target)
best_hgb_pois = search_hgb.best_estimator_
y_pred_hgb_pois = best_hgb_pois.predict(X_test)
print("HGB-Poisson test:", eval_real(y_test, y_pred_hgb_pois))

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best params (HGB-Poisson): {'model__l2_regularization': 3.753893342212558e-07, 'model__learning_rate': 0.04812520518271316, 'model__max_depth': 8, 'model__max_iter': 594, 'model__max_leaf_nodes': 135, 'model__min_samples_leaf': 30}
Best CV MAE: 990.812423232873
HGB-Poisson test: {'RMSE': 1452.5005791613175, 'MAE': 1003.2234876435664, 'R2': 0.12180517504742194}


In [ ]:
# --- Ridge (con TTR)
ridge_base = Ridge(random_state=42)  # Modelo Ridge: regresión lineal con regularización L2

pipe_ridge = Pipeline(steps=[
    ("prep", preprocess),  # Aplica el preprocesamiento (transformaciones numéricas y booleanas)
    ("model", TransformedTargetRegressor(  # Transforma el target con log1p para estabilizar valores grandes
        regressor=ridge_base,              # Modelo base: Ridge
        func=np.log1p,                     # log(1 + y) al entrenar
        inverse_func=np.expm1              # exp(y_pred) - 1 al predecir
    ))
])

param_ridge = {
    "model__regressor__alpha": loguniform(1e-4, 1e3),  # Rango de búsqueda del parámetro alpha
}

search_ridge = RandomizedSearchCV(
    estimator=pipe_ridge,                 # Usa el pipeline completo
    param_distributions=param_ridge,      # Espacio de búsqueda de hiperparámetros
    n_iter=30,                            # Número de combinaciones a probar
    scoring="r2",                         # Métrica: R² (cuánto explica el modelo)
    cv=cv,                                # Validación cruzada (KFold de 5)
    n_jobs=-1,                            # Usa todos los núcleos
    random_state=42,                      # Reproducibilidad
    verbose=1                             # Muestra progreso del entrenamiento
)

search_ridge.fit(X_train, y_train)        # Entrena el modelo con búsqueda aleatoria
print("Best params (Ridge):", search_ridge.best_params_)  # Muestra los mejores hiperparámetros
print("Best CV R²:", search_ridge.best_score_)            # Muestra el R² promedio del CV

best_ridge = search_ridge.best_estimator_  # Obtiene el mejor pipeline ajustado
y_pred_ridge = best_ridge.predict(X_test)  # Realiza predicciones sobre el conjunto de prueba
print("Ridge (tuned) test:", eval_real(y_test, y_pred_ridge))  # Evalúa con RMSE, MAE y R²

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best params (Ridge): {'model__regressor__alpha': 9.047071957568372}
Best CV R²: 0.014321819886957377
Ridge (tuned) test: {'RMSE': 1545.1460491758937, 'MAE': 957.0119408189241, 'R2': 0.006203834686341891}


In [ ]:
# --- Random Forest (rápido) con TTR
rf_base = RandomForestRegressor(n_jobs=-1, random_state=42)  # Modelo base Random Forest (paralelizado)

pipe_rf = Pipeline(steps=[
    ("prep", preprocess),  # Misma transformación de entrada
    ("model", TransformedTargetRegressor(  # Transforma el target con log1p/expm1
        regressor=rf_base,
        func=np.log1p, inverse_func=np.expm1
    ))
])

param_rf_fast = {
    "model__regressor__n_estimators": randint(100, 300),   # Número de árboles
    "model__regressor__max_depth": randint(4, 12),         # Profundidad máxima de cada árbol
    "model__regressor__min_samples_split": randint(2, 8),  # Mínimo de muestras para dividir un nodo
    "model__regressor__min_samples_leaf": randint(1, 5),   # Mínimo de muestras por hoja
    "model__regressor__max_features": ["sqrt", 0.5, 0.7],  # Porcentaje de variables por split
    "model__regressor__bootstrap": [True],                 # Usa muestreo con reemplazo
    "model__regressor__max_samples": [0.5, 0.7],           # Submuestreo para acelerar
}

search_rf = RandomizedSearchCV(
    estimator=pipe_rf,                 # Pipeline con RF
    param_distributions=param_rf_fast, # Espacio de búsqueda
    n_iter=19,                         # 19 combinaciones rápidas
    scoring="r2",                      # Métrica: R²
    cv=2,                              # 2 folds para reducir tiempo
    n_jobs=-1,                         # Paralelización total
    random_state=42,                   # Reproducibilidad
    verbose=1                          # Muestra progreso
)

search_rf.fit(X_train, y_train)        # Entrena el modelo
print("Best params (RF):", search_rf.best_params_)  # Muestra los mejores hiperparámetros
print("Best CV R²:", search_rf.best_score_)         # R² promedio en validación cruzada

best_rf = search_rf.best_estimator_    # Obtiene el mejor modelo
y_pred_rf = best_rf.predict(X_test)    # Predicciones sobre test
print("RandomForest (tuned fast) test:", eval_real(y_test, y_pred_rf))  # Evalúa métricas finales

Fitting 2 folds for each of 19 candidates, totalling 38 fits
Best params (RF): {'model__regressor__bootstrap': True, 'model__regressor__max_depth': 11, 'model__regressor__max_features': 0.7, 'model__regressor__max_samples': 0.7, 'model__regressor__min_samples_leaf': 2, 'model__regressor__min_samples_split': 7, 'model__regressor__n_estimators': 150}
Best CV R²: 0.04549428940550709
RandomForest (tuned fast) test: {'RMSE': 1515.7910440833007, 'MAE': 930.4099014697834, 'R2': 0.04360583327124912}


In [ ]:
# --- XGBoost (rápido) con TTR
xgb_base = XGBRegressor(tree_method="hist", random_state=42, n_jobs=-1)  # XGBoost optimizado con histogramas

pipe_xgb = Pipeline(steps=[
    ("prep", preprocess),  # Preprocesamiento estándar
    ("model", TransformedTargetRegressor(  # Escala logarítmica del target
        regressor=xgb_base,
        func=np.log1p, inverse_func=np.expm1
    ))
])

param_xgb = {
    "model__regressor__n_estimators": randint(300, 800),     # Número de árboles
    "model__regressor__max_depth": randint(4, 9),            # Profundidad de los árboles
    "model__regressor__learning_rate": uniform(0.03, 0.04),  # Tasa de aprendizaje
    "model__regressor__subsample": uniform(0.7, 0.2),        # Submuestreo de filas
    "model__regressor__colsample_bytree": uniform(0.7, 0.2), # Submuestreo de columnas
    "model__regressor__min_child_weight": randint(1, 5),     # Regularización por tamaño mínimo de nodo
    "model__regressor__gamma": uniform(0.0, 0.2),            # Penalización por complejidad
    "model__regressor__reg_lambda": uniform(0.8, 2.0),       # Regularización L2
    "model__regressor__reg_alpha": uniform(0.0, 0.3),        # Regularización L1
}

search_xgb = RandomizedSearchCV(
    estimator=pipe_xgb,                 # Pipeline con XGBoost
    param_distributions=param_xgb,      # Espacio de hiperparámetros
    n_iter=10,                          # 10 combinaciones (rápido)
    scoring="r2",                       # Métrica: R²
    cv=2,                               # 2 folds (rápido)
    n_jobs=-1,                          # Paralelización total
    random_state=42,                    # Reproducible
    verbose=1                           # Muestra progreso
)

search_xgb.fit(X_train, y_train)        # Entrena búsqueda aleatoria
print("Best params (XGB):", search_xgb.best_params_)  # Mejores parámetros
print("Best CV R²:", search_xgb.best_score_)          # Desempeño promedio CV

best_xgb = search_xgb.best_estimator_   # Mejor modelo encontrado
y_pred_xgb = best_xgb.predict(X_test)   # Predicciones sobre test
print("XGBoost (tuned fast) test:", eval_real(y_test, y_pred_xgb))  # Métricas finales

Fitting 2 folds for each of 10 candidates, totalling 20 fits


Best params (XGB): {'model__regressor__colsample_bytree': 0.8510722820635305, 'model__regressor__gamma': 0.08503117489824895, 'model__regressor__learning_rate': 0.03831766651472755, 'model__regressor__max_depth': 7, 'model__regressor__min_child_weight': 2, 'model__regressor__n_estimators': 776, 'model__regressor__reg_alpha': 0.29087538832936755, 'model__regressor__reg_lambda': 2.350265646722229, 'model__regressor__subsample': 0.8878997883128378}
Best CV R²: 0.06730846201919405
XGBoost (tuned fast) test: {'RMSE': 1486.8696341476757, 'MAE': 916.6110279615121, 'R2': 0.07975380741679994}


# ***EVALUACIÓN Y COMPARACIÓN DE MODELOS***

In [231]:
metrics_hgb   = eval_real(y_test, y_pred_hgb_pois)  # Métricas del HGB (Poisson)
metrics_ridge = eval_real(y_test, y_pred_ridge)     # Métricas del Ridge
metrics_rf    = eval_real(y_test, y_pred_rf)        # Métricas del RF
metrics_xgb   = eval_real(y_test, y_pred_xgb)       # Métricas del XGB

# Construye una tabla con los resultados de todos los modelos
comparison_df = pd.DataFrame([
    {"Modelo": "HistGradientBoosting (Poisson)", **metrics_hgb},
    {"Modelo": "Ridge (tuned)", **metrics_ridge},
    {"Modelo": "RandomForest (tuned )", **metrics_rf},
    {"Modelo": "XGBoost (tuned )", **metrics_xgb},
])

# Ordena del mejor (mayor R²) al peor
comparison_df = comparison_df.sort_values(by="R2", ascending=False).reset_index(drop=True)
comparison_df[["RMSE", "MAE", "R2"]] = comparison_df[["RMSE", "MAE", "R2"]].round(4)

print("\n=== Comparación de Modelos (Test) ===")
display(comparison_df)  # Muestra la tabla comparativa


=== Comparación de Modelos (Test) ===


,Modelo,RMSE,MAE,R2
0,HistGradientBoosting (Poisson),1452.5006,1003.2235,0.1218
1,XGBoost (tuned ),1486.8696,916.6110,0.0798
2,RandomForest (tuned ),1515.7910,930.4099,0.0436
3,Ridge (tuned),1545.1460,957.0119,0.0062


# ***GUARDADO DEL MEJOR MODELO***

In [ ]:
import joblib  # Librería para guardar modelos entrenados
joblib.dump(best_hgb_pois, "best_model_hgb_poisson_pipeline.pkl")  # Guarda el mejor pipeline completo
print("Modelo guardado en: best_model_hgb_poisson_pipeline.pkl")   # Confirmación